# Information
## How to
1. Set the parameters. **UPPERCASE** letters are user input variables
2. Run the reprojection cell


# Requirements

In [ ]:
# Standard libraries
import os
import rasterio
import numpy as np

from tqdm import tqdm
from pathlib import Path
from importlib_resources import files

# Custom modules
from beak.experimental.utilities.preparation import impute_data
from beak.experimental.utilities.io import create_file_list, save_raster, check_path


# Scaling

**Scaling** all numerical folders within a specified model configuration.<br>
Reads the <code>ROOT_FOLDER</code> and takes the <code>NUMERICAL</code> subfolder within each model configuration.

**User inputs**

In [ ]:
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 2240
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "us_cont"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")
BASE_EVIDENCE = "geophysics"
  
# Export to current location for testing purposes
PATH_ROOT = BASE_PATH / "PROCESSED" / str("national" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_INPUT = PATH_ROOT / "unified" / BASE_EVIDENCE
PATH_EXPORT = PATH_ROOT / "unified_imputed_mean" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

In [ ]:
# Get the list of folders
root_folder = PATH_INPUT
folders = os.listdir(root_folder)

for folder in folders:
  if os.path.isdir(os.path.join(root_folder, folder)):
    print(folder)
  else:
    print(f"No subfolders existing but found {len(folders)} files.")
    break


In [ ]:
# Select the folders to process
SELECTION = ["gravity",
             "magnetic",
             "magnetotelluric",
             "radiometric",
             "seismic"]

input_folders = [root_folder / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

In [ ]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)
  
print(f"Found {len(file_list)} files.")

**Run**

In [ ]:
# Impute with mean
base_raster = rasterio.open(BASE_RASTER)
base_raster_array = base_raster.read()

for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    array = raster.read()
    
    imputed_array = np.where(array == raster.nodata, np.nan, array)
    imputed_array = impute_data(imputed_array)
    imputed_array = np.where(base_raster_array == base_raster.nodata, np.nan, imputed_array)
    
    output_folder = OUT_FOLDER / str(folder_relative)
    out_path = output_folder / file.name
    check_path(Path(os.path.dirname(out_path)))
    save_raster(out_path, array=imputed_array, dtype="float32", metadata=raster.meta)
    
    